# ML-05 — Feature Vector + Leakage/Privacy Check

**Lane 2 (refresh / opportunity scoring), continuing ML-04.** The data contract in
`w03_data_contract.ipynb` pinned down the windows (B = features only, F = label only), the
dedup rule, and a tentative feature list. This notebook actually **builds** that feature
vector, then does the required honesty work: a **leakage hunt** that attacks my own features
and a **privacy check** on what the pipeline touches.

Built with the `hunting-leakage-and-validating` and `flyrank/flyrank-data` skills.

Every number is computed live from the warehouse release (`FlyRank/internship-warehouse`, build
v20260703). Nothing is hardcoded except release-note cross-checks.

**The one-line claim this notebook earns:** the honest-looking feature set is really only
~0.54–0.64 AUC against a 67% base rate, while the moment a feature reads the label window
(`F`), the score snaps to ~1.0 — so the discipline of keeping features strictly in `B` is
what separates a real model from a memorizer.

## 1. Build the feature vector

Same setup + eligibility as the contract (§0–§1): one row per page (`client_hash_id` ×
`content_hash_id`) at decision date `t`, eligible if `imp_b ≥ 100` and `≥ 15` GSC-available
days in `B`. Features come **only** from `B` (or static `dim_content` descriptors); the label
`declined_30d` comes from `F`. Dedup rule applied (GROUP BY the three key columns).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))

import duckdb
import pandas as pd
import numpy as np
from datetime import timedelta

import hf_query

con = duckdb.connect()
con.execute("CREATE SECRET (TYPE huggingface, TOKEN '" + hf_query.get_token() + "')")

REL = hf_query.REL
T = {
    "content": f"read_parquet('{REL}/dim_content.parquet')",
    "daily":   f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}

D_MAX = con.sql(f"SELECT MAX(report_date) FROM {T['daily']}").fetchone()[0]
t = D_MAX - timedelta(days=30)
b_lo = t - timedelta(days=30)
print("t =", t, "| B = (", b_lo, ",", t, "]")

C:\Users\Bogdan\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


t = 2026-05-31 | B = ( 2026-05-01 , 2026-05-31 ]


In [2]:
feat = con.sql(f"""
WITH win AS (
    SELECT client_hash_id, content_hash_id, report_date,
           gsc_impressions, gsc_clicks, gsc_avg_position, gsc_data_available
    FROM {T['daily']}
    WHERE month IN ('{t:%Y-%m}', '{D_MAX:%Y-%m}')
),
agg AS (
    SELECT client_hash_id, content_hash_id,
           SUM(CASE WHEN report_date <= DATE '{t}' THEN gsc_impressions ELSE 0 END) AS imp_b,
           SUM(CASE WHEN report_date >  DATE '{t}' THEN gsc_impressions ELSE 0 END) AS imp_f,
           SUM(CASE WHEN report_date <= DATE '{t}' THEN gsc_clicks ELSE 0 END) AS clk_b,
           SUM(CASE WHEN report_date <= DATE '{t}' AND gsc_data_available THEN 1 ELSE 0 END) AS gsc_days_b,
           AVG(CASE WHEN report_date <= DATE '{t}' AND gsc_data_available
                    AND gsc_avg_position IS NOT NULL AND gsc_avg_position <> 0
                    THEN gsc_avg_position END) AS pos_avg_b,
           STDDEV(CASE WHEN report_date <= DATE '{t}' AND gsc_data_available
                    AND gsc_avg_position IS NOT NULL AND gsc_avg_position <> 0
                    THEN gsc_avg_position END) AS pos_vol_b
    FROM win GROUP BY 1, 2
),
j AS (
    SELECT a.client_hash_id, a.content_hash_id, a.imp_b, a.imp_f, a.clk_b, a.gsc_days_b,
           a.pos_avg_b, a.pos_vol_b,
           c.content_type, c.word_count, c.char_count, c.keyword_char_count,
           c.keyword_token_count, c.url_char_count, c.main_intent, c.competition_level,
           c.category_count, c.search_volume, c.backlinks,
           c.content_created_date, c.content_updated_date, c.is_deleted, c.is_published,
           c.provider_used, c.model_used
    FROM agg a LEFT JOIN {T['content']} c USING (client_hash_id, content_hash_id)
)
SELECT *,
       DATE '{t}' - content_created_date AS age_days,
       DATE '{t}' - content_updated_date AS days_since_update,
       CASE WHEN imp_f < 0.8 * imp_b THEN 1 ELSE 0 END AS declined_30d
FROM j
WHERE imp_b >= 100 AND gsc_days_b >= 15
""").df()

feat["ctr_b"] = feat.clk_b / feat.imp_b

print("eligible pages:", len(feat))
print("declined_30d base rate:", round(feat.declined_30d.mean(), 4))
print("columns:", list(feat.columns))

eligible pages: 108254
declined_30d base rate: 0.6744
columns: ['client_hash_id', 'content_hash_id', 'imp_b', 'imp_f', 'clk_b', 'gsc_days_b', 'pos_avg_b', 'pos_vol_b', 'content_type', 'word_count', 'char_count', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'main_intent', 'competition_level', 'category_count', 'search_volume', 'backlinks', 'content_created_date', 'content_updated_date', 'is_deleted', 'is_published', 'provider_used', 'model_used', 'age_days', 'days_since_update', 'declined_30d', 'ctr_b']


## 2. Feature notes

For each feature: what it means, how missing values are handled, and **when it exists**.

**Source = `B` / static (features knowable at or before `t`):**

| Feature | Meaning | Missing / scale | Available before `t`? |
|---|---|---|---|
| `imp_b` | impressions summed over B | ≥100 by filter | yes — B |
| `clk_b` | clicks summed over B | fill 0 | yes — B |
| `ctr_b` | `clk_b / imp_b` | fill 0 | yes — B |
| `gsc_days_b` | # GSC-available days in B | ≥15 by filter | yes — B |
| `pos_avg_b` | mean `gsc_avg_position` in B | NULL/0 → missing, fill 0 | yes — B |
| `pos_vol_b` | std of position in B | NULL → fill 0 | yes — B |
| `word_count`, `char_count` | content size | ~22% missing, patterned by type → `has_` flag | yes — static |
| `keyword_char_count`, `keyword_token_count`, `url_char_count` | keyword/url descriptors | mostly complete | yes — static |
| `category_count`, `search_volume`, `backlinks` | demand/competition context | `search_volume` 1.5%, `backlinks` ~31% → `has_` flag | yes — static |
| `content_type` | keyword / feedly / comparison | complete | yes — static |
| `main_intent` | informational / commercial / … | 1.6% missing → fill "unknown" | yes — static |
| `competition_level` | LOW / MEDIUM / HIGH | 2% missing → fill "unknown" | yes — static |
| `age_days`, `days_since_update` | `t − created/updated` | approx. (`updated` is release-time state) | yes — static |

**Categorical handling:** `content_type`, `main_intent`, `competition_level` are label encoded;
the heavy-missing numeric columns get a companion `has_<col>` binary flag so gap-imputation
never sneaks in a category-only effect (contract §3d).

**Why `imp_b` doubles as the scale feature:** the label is `imp_f < 0.8 × imp_b`, so an honest
scale read at `t` is allowed — but the `F` side of that ratio (`imp_f`) **is** the answer and
must never be a feature (proved in §3.1).

In [3]:
base_feats = ["imp_b", "clk_b", "gsc_days_b", "pos_avg_b", "pos_vol_b"]
count_feats = ["word_count", "char_count", "keyword_char_count", "keyword_token_count",
               "url_char_count", "category_count", "search_volume", "backlinks",
               "age_days", "days_since_update"]

legal = feat[base_feats + count_feats + ["ctr_b", "content_type", "main_intent", "competition_level"]].copy()
for c in ["word_count", "search_volume", "backlinks"]:
    legal["has_" + c] = (~feat[c].isna()).astype(int)

print("legal feature set shape:", legal.shape)
print("missing shares in legal set:")
for c in legal.columns:
    m = legal[c].isna().mean()
    if m > 0:
        print(f"   {c:24s} {m:.3f}")

legal feature set shape: (108254, 22)
missing shares in legal set:
   word_count               0.224
   char_count               0.224
   search_volume            0.015
   backlinks                0.308
   main_intent              0.016
   competition_level        0.020


## 3. The leakage hunt

Attack my own features with the three-way taxonomy from the skill: **(1) label-derived / future
windows**, **(2) overlapping windows**, **(3) decision-derived product flags** — plus the split
honesty check (random vs grouped). Constructed deliberately: a model is trained on the **legal**
set, then on the same set with each leak **added**, and the score jump is the confession.
Base rate 0.674 is printed beside every model.

### 3.1 Test 1 — label-derived / future-window leaks (`imp_f`, `imp_b`)

The label is `imp_f < 0.8 × imp_b`. If I hand the model `imp_f` (or `imp_b`), it has the answer.
The test: add them and watch AUC → 1.0.

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.metrics import precision_score, roc_auc_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import make_pipeline

y = feat.declined_30d.values

def encode(Xdf):
    Xdf = Xdf.copy()
    for c in Xdf.select_dtypes(include=["object", "str"]).columns:
        Xdf[c] = LabelEncoder().fit_transform(Xdf[c].astype(str))
    return Xdf.fillna(0)

def pipe():
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))

def eval_split(X, seed=0):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=seed, stratify=y)
    m = pipe().fit(Xtr, ytr)
    return roc_auc_score(yte, m.predict_proba(Xte)[:, 1]), precision_score(yte, m.predict(Xte))

Xlegal = encode(legal)

leak1 = legal.copy()
leak1["imp_f"] = feat.imp_f      # the label's own numerator -> the answer
Xleak1 = encode(leak1)

print("base rate                 :", round(y.mean(), 4))
print("legal (B-only)            : AUC", round(eval_split(Xlegal)[0], 4),
      "| prec", round(eval_split(Xlegal)[1], 4))
print("+ imp_f (label-derived)   : AUC", round(eval_split(Xleak1)[0], 4),
      "| prec", round(eval_split(Xleak1)[1], 4))

base rate                 : 0.6744


legal (B-only)            : AUC 0.6325 | prec 0.6856


+ imp_f (label-derived)   : AUC 0.9976 | prec 0.9237


**Reading:** legal AUC ≈ 0.64 against a 0.674 base rate is barely above chance. Adding the
label's own input (`imp_f`) snaps AUC to **≈ 1.0**. That is the signature of label leakage — a
near-perfect score is the confession, not the win. `imp_f` (and any column derived from `F`, or
from a window straddling `t`) is **excluded**; only `imp_b` stays, because it is knowable at `t`.

### 3.2 Test 2 — overlapping / future window (`q90` style)

The q90 table's fixed window straddles `t` (contract §3f). To show the *mechanism* without
touching q90, I inject a stand-in forward signal (`imp_f > imp_b`) and watch the score climb.

In [5]:
fut = legal.copy()
fut["fwd_imp_signal"] = (feat.imp_f > feat.imp_b).astype(int)
Xfut = encode(fut)
print("base rate           :", round(y.mean(), 4))
print("legal (B-only)      : AUC", round(eval_split(Xlegal)[0], 4))
print("+ forward-window sig : AUC", round(eval_split(Xfut)[0], 4),
      "| prec", round(eval_split(Xfut)[1], 4))

base rate           : 0.6744


legal (B-only)      : AUC 0.6325


+ forward-window sig : AUC 0.8728 | prec 0.8651


**Reading:** a feature computed over `F` (the q90 window contains exactly this span) pushes AUC
from ≈ 0.64 to ≈ 0.88. Feels great, but it is **future information** — a model that uses it
cannot exist at `t`. This is why the whole q90 table is excluded (§4).

### 3.3 Test 3 — decision-derived product flags (`provider_used`, `model_used`)

These record *how content was produced* — a decision already made by an existing system. Using
them as features means recovering the old rule, not the world (circular). They could form a
baseline to beat, never inputs.

In [6]:
prod = legal.copy()
prod["provider_used"] = feat.provider_used.astype(str)
prod["model_used"] = feat.model_used.astype(str)
Xprod = encode(prod)
print("legal               : AUC", round(eval_split(Xlegal)[0], 4))
print("+ provider/model     : AUC", round(eval_split(Xprod)[0], 4),
      "| prec", round(eval_split(Xprod)[1], 4))
print("provider_used distinct values:", feat.provider_used.nunique())

legal               : AUC 0.6325


+ provider/model     : AUC 0.6515 | prec 0.6923
provider_used distinct values: 6


**Reading:** even when they add a little score, they are excluded on *principle*: their
information is a past production decision, and — importantly — `provider_used` / `model_used`
carry free-text that can hold non-anonymized labels. They are excluded for **both** decision-
derived and **privacy** reasons (§4). None of those raw values is printed here.

### 3.4 Test 4 — split honesty: random vs grouped by client

Rows from one client share hidden character. A random split lets the model memorize the client
and fake skill; the honest question is "does it work on a client it never saw?".

In [7]:
gkf = GroupKFold(n_splits=5)
aucs, precs = [], []
for tr, te in gkf.split(Xlegal, y, groups=feat.client_hash_id.values):
    mm = pipe().fit(Xlegal.iloc[tr], y[tr])
    ya = mm.predict_proba(Xlegal.iloc[te])[:, 1]
    yp = mm.predict(Xlegal.iloc[te])
    aucs.append(roc_auc_score(y[te], ya))
    precs.append(precision_score(y[te], yp))

print("base rate                      :", round(y.mean(), 4))
print("random split  AUC / prec       :", round(eval_split(Xlegal)[0], 4),
      "/", round(eval_split(Xlegal)[1], 4))
print("grouped by client AUC / prec   :", round(np.mean(aucs), 4),
      "/", round(np.mean(precs), 4))
print("AUC gap (random - grouped)     :", round(eval_split(Xlegal)[0] - np.mean(aucs), 4))

base rate                      : 0.6744


random split  AUC / prec       : 0.6325 / 0.6856
grouped by client AUC / prec   : 0.5385 / 0.6763


AUC gap (random - grouped)     : 0.0939


**Reading:** the grouped (client-holdout) AUC is ~0.54, well under the random-split ~0.64. The
GAP itself is the finding: the random split was partly memorizing clients. Everything downstream
must use a **grouped / client-holdout** split for honest numbers — same conclusion as the
contract's 03_train_model.py. Keep the split grouped and report base rate beside every metric.

In [8]:
summary = pd.DataFrame({
    "model": ["base rate", "legal (B-only)", "+ imp_f (label-derived)",
              "+ forward-window signal", "+ provider/model (product flags)", "random split",
              "grouped by client"],
    "AUC": [round(y.mean(), 3), round(eval_split(Xlegal)[0], 3), round(eval_split(Xleak1)[0], 3),
            round(eval_split(Xfut)[0], 3), round(eval_split(Xprod)[0], 3),
            round(eval_split(Xlegal)[0], 3), round(np.mean(aucs), 3)],
})
summary

,model,AUC
0,base rate,0.674
1,legal (B-only),0.632
2,+ imp_f (label-derived),0.998
3,+ forward-window signal,0.873
4,+ provider/model (product flags),0.651
5,random split,0.632
6,grouped by client,0.539


## 4. What I excluded and why

| Field(s) | Why excluded |
|---|---|
| `imp_f` and any `F`-derived aggregate | IS the label's numerator (§3.1) — future information |
| `fact_content_query_90d` (all cols) | fixed window straddles `t` → contains `F` (§3.2) — future information |
| `provider_used`, `model_used` | decision-derived product flags (§3.3) — circular; may carry non-anonymized labels (privacy) |
| `ga4_*`, `sessions_*`, `ai_*`, `scroll_events` | zero-filled outside GA4 availability (~74% off) — filler, not "no engagement" (contract §3e) |
| `last_optimized_date`, `optimization_eligible_date` | product-workflow timestamps, release-time state |
| `keyword_hash_id`, `url_hash_id` | high-cardinality IDs, nothing generalizable |
| `is_deleted`, `is_published` | context filters only (population), not features |
| `client_hash_id`, `content_hash_id` | pseudonymous IDs — grouping/splitting only, never features (privacy) |

**Privacy note:** the pipeline *reads* `provider_used` / `model_used` / IDs only to classify and
join — those values never enter the model, the report, or any output. Nothing client-identifying
is printed or written anywhere in `work/`.

## Self-check

- [x] Feature vector built with executed queries; features strictly in `B` or static
- [x] Leakage tests run for all three taxonomy arms + split honesty
- [x] Base rate (0.674) printed beside every metric
- [x] Excluded list carries a one-line why for every field
- [x] No client names, URLs, provider values, or raw identifiers in any output — aggregates only
- [x] Claims use careful words: observed, measured, directional, decision-support
- [ ] The notebook runs top to bottom with no errors (Kernel → Restart & Run all)
- [ ] Committed to `work/notebooks/` — then submit repo URL on the ML-05 card